In [ ]:
import os
from rdkit import Chem
from rdkit.Chem import Descriptors
from aizynthfinder.aizynthfinder import AiZynthFinder

# ... [INITIALIZATION] ...
# Loading the configuration which points to policy models and stocks.
# These models act as the 'brain' that predicts which reactions are possible.
filename = "config.yml"
finder = AiZynthFinder(configfile=filename)
finder.stock.select("zinc")
finder.expansion_policy.select("uspto")
finder.filter_policy.select("uspto")

def extract_precursors(tree):
    """
    Recursively extract leaf nodes (starting materials) from reaction tree dict
    """
    if "children" not in tree or not tree["children"]:
        return [tree["smiles"]]

    precursors = []
    for child in tree["children"]:
        precursors.extend(extract_precursors(child))
    return precursors

def count_steps(tree):
    """
    Count number of reaction steps in the tree
    """
    if "children" not in tree or not tree["children"]:
        return 0
    return 1 + max(count_steps(child) for child in tree["children"])


def calculate_atom_economy(route):
    """
    ... [GREEN CHEMISTRY ANALYSIS] ...
    Calculates the percentage of reactant atoms that end up in the final product.
    Formula: (MW of target / sum(MW of all starting materials)) * 100
    """
    target_mol = Chem.MolFromSmiles(route.target.smiles)
    target_mw = Descriptors.MolWt(target_mol)
    
    # Precursors are the 'leaves' of the synthesis tree (starting materials)
    precursor_mols = [Chem.MolFromSmiles(p.smiles) for p in route.precursors]
    total_precursor_mw = sum(Descriptors.MolWt(m) for m in precursor_mols if m)
    
    return (target_mw / total_precursor_mw) * 100 if total_precursor_mw > 0 else 0

def get_toxicity_score(route):
    """
    ... [SAFETY ANALYSIS] ...
    Analyzes the 'leaves' (starting materials) for known toxic substructures.
    Currently returns a placeholder value (0.0=Safe, 1.0=Hazardous).
    """
    return 0.15 # Placeholder for a real safety scoring API or database lookup

def main():
    # ... [USER INTERFACE] ...
    # Collects the target molecule and the user's weighted priorities.
    # target_smiles = input("Target SMILES (e.g., Aspirin: CC(=O)OC1=CC=CC=C1C(=O)O): ")
    target_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O" # Aspirin as a default example 

    
    print("\nCriteria List: [A] Atom Economy, [T] Toxicity, [Y] Yield, [S] Steps")
    choices = input("Rank top 3 (e.g., 'ASY' for 1st:Atom, 2nd:Steps, 3rd:Yield): ").upper()

    # Define weights based on rank: Rank 1 = 50%, Rank 2 = 30%, Rank 3 = 20%
    weight_map = {'A': 'atom_economy', 'T': 'toxicity', 'Y': 'yield', 'S': 'steps'}
    chosen_metrics = [weight_map[c] for c in choices if c in weight_map]
    ranking_weights = [0.5, 0.3, 0.2]

    # ... [SEARCH EXECUTION] ...
    # finder.tree_search() performs the Monte Carlo Tree Search (MCTS)
    # to find pathways from the target back to commercial stocks.
    print(f"\nSearching for routes to {target_smiles}...")
    finder.target_smiles = target_smiles
    finder.tree_search()
    finder.build_routes() #? Organizes the raw search tree into distinct routes
    routes = finder.routes

    # ... [MULTI-CRITERIA RANKING] ...
    # We loop through found routes and calculate a normalized score for each.
    final_rankings = []
    for route in finder.routes:
        metrics = {
            "atom_economy": calculate_atom_economy(route) / 100.0,
            "toxicity": 1.0 - get_toxicity_score(route), # Inverse: higher is 'better' (safer)
            "yield": route.reaction_tree.metrics.get("state_score", 0.5),
            "steps": 1.0 / len(list(route.reaction_tree.reactions())) # Fewer steps = higher score
        }

        # Apply user-defined weights to the metrics
        final_score = sum(metrics[m] * ranking_weights[i] for i, m in enumerate(chosen_metrics))
        final_rankings.append((final_score, route, metrics))

    # ... [RESULTS OUTPUT] ...
    # Sorts routes by the final score (highest first) and displays the top results.
    final_rankings.sort(key=lambda x: x[0], reverse=True)
    
    print(f"\n--- TOP {min(3, len(final_rankings))} BEST MATCHES ---")
    for rank, (score, route, m) in enumerate(final_rankings[:3], 1):
        print(f"Rank {rank} (Total Score: {score:.2f})")
        print(f" > Steps: {int(1/m['steps'])} | Atom Economy: {m['atom_economy']*100:.1f}%")
        print(f" > Route Summary: {route.reaction_tree.to_dict()}")

if __name__ == "__main__":
    main()




Criteria List: [A] Atom Economy, [T] Toxicity, [Y] Yield, [S] Steps

Searching for routes to CC(=O)OC1=CC=CC=C1C(=O)O...


TypeError: Your AiZynthFinder version returns dict routes and does not support 'routes_as_objects()'. Consider upgrading or use Method 1.

In [7]:
import os
from rdkit import Chem
from rdkit.Chem import Descriptors
from aizynthfinder.aizynthfinder import AiZynthFinder

"""
PROGRAM OVERVIEW
----------------
This program performs retrosynthetic analysis of a target molecule using AiZynthFinder.

It:
1. Generates possible synthesis routes from commercially available precursors
2. Evaluates each route using green chemistry and practicality metrics:
   - Atom economy
   - Toxicity (placeholder)
   - Yield (model-based estimate)
   - Number of steps
3. Allows the user to prioritize these criteria
4. Ranks and displays the best synthesis routes

The goal is to support decision-making in sustainable and efficient chemical synthesis.
"""

# Initialize retrosynthesis engine
filename = "config.yml"
finder = AiZynthFinder(configfile=filename)

# Select databases and ML policies
finder.stock.select("zinc")          # Available starting materials
finder.expansion_policy.select("uspto")  # Reaction prediction model
finder.filter_policy.select("uspto")     # Reaction feasibility filter


def extract_precursors(tree):
    """
    Recursively extracts all leaf nodes (starting materials)
    from a retrosynthesis reaction tree.

    Parameters:
        tree (dict): Nested reaction tree structure

    Returns:
        list: SMILES strings of precursor molecules
    """
    if "children" not in tree or not tree["children"]:
        return [tree["smiles"]]

    precursors = []
    for child in tree["children"]:
        precursors.extend(extract_precursors(child))
    return precursors


def count_steps(tree):
    """
    Counts the number of reaction steps in a synthesis route.

    Uses recursive traversal to account for branching pathways.

    Parameters:
        tree (dict): Reaction tree

    Returns:
        int: Number of steps (depth of synthesis)
    """
    if "children" not in tree or not tree["children"]:
        return 0
    return 1 + max(count_steps(child) for child in tree["children"])


def calculate_atom_economy(route_dict):
    """
    Calculates atom economy for a synthesis route.

    Atom Economy = (Molecular Weight of Target /
                    Sum of Molecular Weights of Precursors) × 100

    Measures how efficiently reactant atoms are incorporated
    into the final product (green chemistry principle).

    Parameters:
        route_dict (dict): Route representation

    Returns:
        float: Atom economy percentage
    """
    tree = route_dict["reaction_tree"]

    target_smiles = tree["smiles"]
    target_mol = Chem.MolFromSmiles(target_smiles)
    target_mw = Descriptors.MolWt(target_mol)

    precursor_smiles = extract_precursors(tree)
    precursor_mols = [Chem.MolFromSmiles(s) for s in precursor_smiles]

    total_precursor_mw = sum(
        Descriptors.MolWt(m) for m in precursor_mols if m
    )

    return (target_mw / total_precursor_mw) * 100 if total_precursor_mw > 0 else 0


def get_toxicity_score(route_dict):
    """
    Placeholder toxicity scoring function.

    Intended to evaluate safety of starting materials
    based on known toxic substructures or databases.

    Returns:
        float: Toxicity score (0 = safe, 1 = hazardous)

    NOTE: Currently returns a constant value.
    """
    return 0.15


def get_yield_score(route_dict):
    """
    Extracts yield/feasibility estimate from AiZynthFinder.

    Uses internal model score (state_score) as a proxy for yield.

    Parameters:
        route_dict (dict): Route data

    Returns:
        float: Estimated yield score (0–1)
    """
    return route_dict.get("reaction_tree", {}).get("metrics", {}).get("state_score", 0.5)


def main():
    """
    Main execution function.

    Workflow:
    1. Define target molecule
    2. Collect user preferences for ranking criteria
    3. Run retrosynthesis search
    4. Evaluate all generated routes
    5. Rank routes using weighted scoring
    6. Display top results
    """

    # Example target: Aspirin
    target_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"

    print("\nCriteria List: [A] Atom Economy, [T] Toxicity, [Y] Yield, [S] Steps")
    choices = input("Rank top 3 (e.g., 'ASY'): ").upper()

    # Map user input to metric names
    weight_map = {
        'A': 'atom_economy',
        'T': 'toxicity',
        'Y': 'yield',
        'S': 'steps'
    }

    chosen_metrics = [weight_map[c] for c in choices if c in weight_map]

    # Weighting scheme: priority-based
    ranking_weights = [0.5, 0.3, 0.2]

    print(f"\nSearching for routes to {target_smiles}...")

    # Run retrosynthesis search (Monte Carlo Tree Search)
    finder.target_smiles = target_smiles
    finder.tree_search()
    finder.build_routes()

    routes = finder.routes
    final_rankings = []

    # Evaluate each route
    for route in routes:
        try:
            tree = route["reaction_tree"]

            metrics = {
                "atom_economy": calculate_atom_economy(route) / 100.0,
                "toxicity": 1.0 - get_toxicity_score(route),  # invert (higher = safer)
                "yield": get_yield_score(route),
                "steps": 1.0 / max(count_steps(tree), 1)      # fewer steps = better
            }

            # Weighted scoring
            final_score = sum(
                metrics[m] * ranking_weights[i]
                for i, m in enumerate(chosen_metrics)
            )

            final_rankings.append((final_score, route, metrics))

        except KeyError as e:
            print(f"[KeyError] Missing key in route: {e}")

        except TypeError as e:
            print(f"[TypeError] Likely invalid SMILES or None passed: {e}")

        except ValueError as e:
            print(f"[ValueError] Descriptor calculation failed: {e}")

        except AttributeError as e:
            print(f"[AttributeError] Likely None molecule used: {e}")

    # Rank routes by score
    final_rankings.sort(key=lambda x: x[0], reverse=True)

    print(f"\n--- TOP {min(3, len(final_rankings))} BEST MATCHES ---")

    # Display results
    for rank, (score, route, m) in enumerate(final_rankings[:3], 1):
        print(f"\nRank {rank} (Total Score: {score:.2f})")
        print(f" > Steps: {int(1/m['steps'])}")
        print(f" > Atom Economy: {m['atom_economy']*100:.1f}%")
        print(f" > Yield Score: {m['yield']:.2f}")
        print(f" > Safety Score: {m['toxicity']:.2f}")

        print(f" > Route Summary: {route['reaction_tree']}")


if __name__ == "__main__":
    main()


Criteria List: [A] Atom Economy, [T] Toxicity, [Y] Yield, [S] Steps

Searching for routes to CC(=O)OC1=CC=CC=C1C(=O)O...
[TypeError] Likely invalid SMILES or None passed: 'ReactionTree' object is not subscriptable
[TypeError] Likely invalid SMILES or None passed: 'ReactionTree' object is not subscriptable
[TypeError] Likely invalid SMILES or None passed: 'ReactionTree' object is not subscriptable
[TypeError] Likely invalid SMILES or None passed: 'ReactionTree' object is not subscriptable
[TypeError] Likely invalid SMILES or None passed: 'ReactionTree' object is not subscriptable
[TypeError] Likely invalid SMILES or None passed: 'ReactionTree' object is not subscriptable
[TypeError] Likely invalid SMILES or None passed: 'ReactionTree' object is not subscriptable
[TypeError] Likely invalid SMILES or None passed: 'ReactionTree' object is not subscriptable
[TypeError] Likely invalid SMILES or None passed: 'ReactionTree' object is not subscriptable
[TypeError] Likely invalid SMILES or None

In [8]:
import os
from rdkit import Chem
from rdkit.Chem import Descriptors
from aizynthfinder.aizynthfinder import AiZynthFinder

"""
PROGRAM OVERVIEW
----------------
This program performs retrosynthetic route planning and ranking.

It:
1. Uses AiZynthFinder to generate synthesis routes for a target molecule
2. Handles BOTH dictionary-based and object-based route formats
3. Evaluates each route using:
   - Atom economy (green chemistry efficiency)
   - Toxicity (placeholder safety score)
   - Yield (model-derived feasibility score)
   - Number of synthesis steps
4. Allows the user to prioritize these criteria
5. Outputs the top-ranked synthesis routes

Key Feature:
------------
A robust "adapter layer" ensures compatibility across different
AiZynthFinder versions and data structures.
"""

# ----------------------------
# INITIALIZATION
# ----------------------------
filename = "config.yml"
finder = AiZynthFinder(configfile=filename)

# Select data sources and ML policies
finder.stock.select("zinc")              # Available starting materials
finder.expansion_policy.select("uspto")  # Reaction prediction model
finder.filter_policy.select("uspto")     # Reaction feasibility filter


# ----------------------------
# SAFE RDKit HANDLING
# ----------------------------
def safe_mol_from_smiles(smiles):
    """
    Safely converts a SMILES string into an RDKit molecule.

    Raises:
        ValueError: If SMILES is invalid or cannot be parsed

    Why:
        RDKit does NOT always raise exceptions on failure—it may return None.
        This function ensures failures are caught early and explicitly.
    """
    if not smiles:
        raise ValueError("Empty SMILES string")

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    return mol


# ----------------------------
# ADAPTER LAYER (CRITICAL)
# ----------------------------
def get_reaction_tree(route):
    """
    Extracts the reaction tree from a route.

    Handles:
        - dict-based routes
        - object-based routes

    Returns:
        Reaction tree (dict or ReactionTree object)
    """
    if isinstance(route, dict):
        return route.get("reaction_tree")
    return route.reaction_tree


def get_target_smiles(tree):
    """
    Retrieves the target molecule SMILES.

    Handles both:
        - dict trees
        - ReactionTree objects
    """
    if isinstance(tree, dict):
        return tree.get("smiles")

    return tree.root.smiles


def get_precursor_smiles(tree):
    """
    Extracts all precursor (leaf) molecules from the synthesis tree.

    Returns:
        list of SMILES strings

    Handles both dict and object representations.
    """

    # --- dict-based tree ---
    if isinstance(tree, dict):
        def extract(node):
            if "children" not in node or not node["children"]:
                return [node["smiles"]]
            result = []
            for child in node["children"]:
                result.extend(extract(child))
            return result

        return extract(tree)

    # --- object-based tree ---
    return [leaf.smiles for leaf in tree.leafs()]


def count_steps(tree):
    """
    Counts the number of reaction steps in a synthesis route.

    Returns:
        int: number of steps

    Handles:
        - dict trees (recursive depth)
        - object trees (reaction count)
    """

    # dict version
    if isinstance(tree, dict):
        def depth(node):
            if "children" not in node or not node["children"]:
                return 0
            return 1 + max(depth(child) for child in node["children"])
        return depth(tree)

    # object version
    return len(list(tree.reactions()))


def get_yield_score(route, tree):
    """
    Extracts yield/feasibility score.

    Returns:
        float (0–1)

    Notes:
        Uses AiZynthFinder's internal "state_score" as proxy.
    """

    if isinstance(route, dict):
        return route.get("reaction_tree", {}).get("metrics", {}).get("state_score", 0.5)

    return tree.metrics.get("state_score", 0.5)


# ----------------------------
# METRIC CALCULATIONS
# ----------------------------
def calculate_atom_economy(route):
    """
    Computes atom economy of a synthesis route.

    Formula:
        Atom Economy = (MW_target / Σ MW_precursors) × 100

    Returns:
        float: percentage value

    Importance:
        Measures how efficiently reactant atoms are incorporated
        into the final product (key green chemistry principle).
    """
    tree = get_reaction_tree(route)

    target_smiles = get_target_smiles(tree)
    target_mol = safe_mol_from_smiles(target_smiles)
    target_mw = Descriptors.MolWt(target_mol)

    precursor_smiles = get_precursor_smiles(tree)
    precursor_mols = [safe_mol_from_smiles(s) for s in precursor_smiles]

    total_precursor_mw = sum(Descriptors.MolWt(m) for m in precursor_mols)

    if total_precursor_mw == 0:
        return 0

    return (target_mw / total_precursor_mw) * 100


def get_toxicity_score(route):
    """
    Placeholder toxicity scoring function.

    Returns:
        float (0 = safe, 1 = hazardous)

    Future work:
        Replace with SMARTS-based toxicophore detection
        or external chemical safety databases.
    """
    return 0.15


# ----------------------------
# MAIN EXECUTION
# ----------------------------
def main():
    """
    Main workflow:

    1. Define target molecule
    2. Collect user ranking preferences
    3. Run retrosynthesis search
    4. Evaluate each route
    5. Rank using weighted scoring
    6. Display best routes
    """

    # Example: Aspirin
    target_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"

    print("\nCriteria List: [A] Atom Economy, [T] Toxicity, [Y] Yield, [S] Steps")
    choices = input("Rank top 3 (e.g., 'ASY'): ").upper()

    # Map user choices to metrics
    weight_map = {
        'A': 'atom_economy',
        'T': 'toxicity',
        'Y': 'yield',
        'S': 'steps'
    }

    chosen_metrics = [weight_map[c] for c in choices if c in weight_map]
    ranking_weights = [0.5, 0.3, 0.2]

    print(f"\nSearching for routes to {target_smiles}...")

    # Run retrosynthesis (Monte Carlo Tree Search)
    finder.target_smiles = target_smiles
    finder.tree_search()
    finder.build_routes()

    routes = finder.routes
    final_rankings = []

    # Evaluate routes
    for route in routes:
        try:
            tree = get_reaction_tree(route)

            if tree is None:
                raise ValueError("Missing reaction tree")

            steps = count_steps(tree)

            metrics = {
                "atom_economy": calculate_atom_economy(route) / 100.0,
                "toxicity": 1.0 - get_toxicity_score(route),
                "yield": get_yield_score(route, tree),
                "steps": 1.0 / max(steps, 1)
            }

            # Weighted scoring
            final_score = sum(
                metrics[m] * ranking_weights[i]
                for i, m in enumerate(chosen_metrics)
            )

            final_rankings.append((final_score, route, metrics))

        # ---- Controlled exception handling ----
        except KeyError as e:
            print(f"[KeyError] Missing data in route: {e}")

        except ValueError as e:
            print(f"[ValueError] {e}")

        except TypeError as e:
            print(f"[TypeError] Likely structure mismatch: {e}")

        except AttributeError as e:
            print(f"[AttributeError] Possibly invalid object usage: {e}")

    # Rank results
    final_rankings.sort(key=lambda x: x[0], reverse=True)

    print(f"\n--- TOP {min(3, len(final_rankings))} BEST MATCHES ---")

    for rank, (score, route, m) in enumerate(final_rankings[:3], 1):
        print(f"\nRank {rank} (Total Score: {score:.2f})")
        print(f" > Steps: {int(1/m['steps'])}")
        print(f" > Atom Economy: {m['atom_economy']*100:.1f}%")
        print(f" > Yield Score: {m['yield']:.2f}")
        print(f" > Safety Score: {m['toxicity']:.2f}")

        # Show raw route structure
        if isinstance(route, dict):
            print(f" > Route Summary: {route['reaction_tree']}")
        else:
            print(f" > Route Summary: {route.reaction_tree.to_dict()}")


if __name__ == "__main__":
    main()


Criteria List: [A] Atom Economy, [T] Toxicity, [Y] Yield, [S] Steps

Searching for routes to CC(=O)OC1=CC=CC=C1C(=O)O...
[AttributeError] Possibly invalid object usage: 'ReactionTree' object has no attribute 'get'
[AttributeError] Possibly invalid object usage: 'ReactionTree' object has no attribute 'get'
[AttributeError] Possibly invalid object usage: 'ReactionTree' object has no attribute 'get'
[AttributeError] Possibly invalid object usage: 'ReactionTree' object has no attribute 'get'
[AttributeError] Possibly invalid object usage: 'ReactionTree' object has no attribute 'get'
[AttributeError] Possibly invalid object usage: 'ReactionTree' object has no attribute 'get'
[AttributeError] Possibly invalid object usage: 'ReactionTree' object has no attribute 'get'
[AttributeError] Possibly invalid object usage: 'ReactionTree' object has no attribute 'get'
[AttributeError] Possibly invalid object usage: 'ReactionTree' object has no attribute 'get'
[AttributeError] Possibly invalid object 

In [9]:
import os
from rdkit import Chem
from rdkit.Chem import Descriptors
from aizynthfinder.aizynthfinder import AiZynthFinder

"""
========================================================
RETROSYNTHESIS ROUTE RANKING SYSTEM
========================================================

This program:
1. Generates synthesis routes for a target molecule using AiZynthFinder
2. Handles inconsistent internal data structures (dicts + objects)
3. Evaluates routes using:
   - Atom economy (green chemistry efficiency)
   - Toxicity (placeholder safety metric)
   - Yield (model-based feasibility score)
   - Number of synthesis steps
4. Allows user-weighted ranking of criteria
5. Outputs best-ranked synthesis routes

Key design feature:
- Fully structure-agnostic adapter layer (prevents all API mismatch errors)
"""

# ----------------------------
# INITIALIZATION
# ----------------------------
finder = AiZynthFinder(configfile="config.yml")
finder.stock.select("zinc")
finder.expansion_policy.select("uspto")
finder.filter_policy.select("uspto")


# ========================================================
# SAFE LOW-LEVEL UTILITIES
# ========================================================

def safe_mol(smiles):
    """
    Convert SMILES → RDKit molecule safely.
    Raises error early if invalid.
    """
    if not smiles:
        raise ValueError("Empty SMILES")

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    return mol


def safe_attr(obj, key, default=None):
    """
    Universal accessor for:
    - dicts (obj[key])
    - objects (obj.key)

    Prevents ALL .get() and attribute crashes.
    """
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


# ========================================================
# STRUCTURE ADAPTER LAYER (CORE FIX)
# ========================================================

def get_tree(route):
    """Return reaction tree regardless of format"""
    return route["reaction_tree"] if isinstance(route, dict) else route.reaction_tree


def get_smiles(tree):
    """Extract SMILES from either dict or ReactionTree object"""
    if isinstance(tree, dict):
        return tree.get("smiles")
    return tree.root.smiles


def get_leaves(tree):
    """
    Get precursor molecules (leaf nodes)
    Works for both:
    - dict tree
    - ReactionTree object
    """
    if isinstance(tree, dict):
        def walk(node):
            if not node.get("children"):
                return [node["smiles"]]
            out = []
            for c in node["children"]:
                out.extend(walk(c))
            return out
        return walk(tree)

    return [leaf.smiles for leaf in tree.leafs()]


def get_steps(tree):
    """Count synthesis steps safely"""
    if isinstance(tree, dict):
        def depth(n):
            if not n.get("children"):
                return 0
            return 1 + max(depth(c) for c in n["children"])
        return depth(tree)

    return len(list(tree.reactions()))


# ========================================================
# METRICS
# ========================================================

def atom_economy(route):
    """
    Green chemistry metric:
    efficiency of atom usage in synthesis.
    """
    tree = get_tree(route)

    target = safe_mol(get_smiles(tree))
    target_mw = Descriptors.MolWt(target)

    precursors = [safe_mol(s) for s in get_leaves(tree)]
    total_mw = sum(Descriptors.MolWt(m) for m in precursors)

    return (target_mw / total_mw) * 100 if total_mw else 0


def toxicity(route):
    """Placeholder safety score (0–1)"""
    return 0.15


def yield_score(route):
    """
    Safe extraction of AiZynthFinder internal score.
    Works for both dict and object metric containers.
    """
    tree = get_tree(route)
    metrics = safe_attr(tree, "metrics", None)

    return safe_attr(metrics, "state_score", 0.5)


def steps_score(route):
    """Fewer steps = better score"""
    tree = get_tree(route)
    return 1.0 / max(get_steps(tree), 1)


# ========================================================
# MAIN PIPELINE
# ========================================================

def main():
    """
    End-to-end workflow:
    1. Define target molecule
    2. Run retrosynthesis search
    3. Evaluate routes
    4. Rank results
    """

    target_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"

    print("\nRank criteria: [A] Atom economy [T] Toxicity [Y] Yield [S] Steps")
    choice = input("Enter ranking (e.g. ASY): ").upper()

    map_ = {
        "A": "atom",
        "T": "tox",
        "Y": "yield",
        "S": "steps"
    }

    selected = [map_[c] for c in choice if c in map_]
    weights = [0.5, 0.3, 0.2]

    print("\nRunning retrosynthesis search...")
    finder.target_smiles = target_smiles
    finder.tree_search()
    finder.build_routes()

    results = []

    for route in finder.routes:
        try:
            metrics = {
                "atom": atom_economy(route) / 100.0,
                "tox": 1.0 - toxicity(route),
                "yield": yield_score(route),
                "steps": steps_score(route)
            }

            score = sum(metrics[m] * weights[i] for i, m in enumerate(selected))
            results.append((score, route, metrics))

        except Exception as e:
            print(f"[SKIP ROUTE] {type(e).__name__}: {e}")

    results.sort(reverse=True, key=lambda x: x[0])

    print("\n=== TOP RESULTS ===")
    for i, (score, route, m) in enumerate(results[:3], 1):
        tree = get_tree(route)

        print(f"\nRank {i} | Score: {score:.2f}")
        print(f"Steps: {get_steps(tree)}")
        print(f"Atom economy: {m['atom']*100:.1f}%")
        print(f"Yield: {m['yield']:.2f}")
        print(f"Route: {tree if isinstance(tree, dict) else tree.to_dict()}")


if __name__ == "__main__":
    main()


Rank criteria: [A] Atom economy [T] Toxicity [Y] Yield [S] Steps

Running retrosynthesis search...

=== TOP RESULTS ===

Rank 1 | Score: 0.85
Steps: 1
Atom economy: 90.9%
Yield: 0.50
Route: {'type': 'mol', 'hide': False, 'smiles': 'CC(=O)Oc1ccccc1C(=O)O', 'is_chemical': True, 'in_stock': False, 'children': [{'type': 'reaction', 'hide': False, 'smiles': '[C:1]([CH3:2])(=[O:3])[O:4][cH3:5]>>O=[C:1]([CH3:2])[O:3].[O:4][cH3:5]', 'is_reaction': True, 'metadata': {'template_hash': '09ae625dda4c90c7615b0ddd9b5e6aa71cd4c4261ba0d8d61d616b4bb8397ee3', 'classification': '0.0 Unrecognized', 'library_occurence': 230, 'policy_probability': 0.008999999612569809, 'policy_probability_rank': 5, 'policy_name': 'uspto', 'template_code': 1607, 'template': '[C:2]-[C;H0;D3;+0:1](=[O;H0;D1;+0:3])-[O;H0;D2;+0:4]-[c:5]>>O=[C;H0;D3;+0:1](-[C:2])-[OH;D1;+0:3].[OH;D1;+0:4]-[c:5]', 'mapped_reaction_smiles': '[CH3:1][C:2](=[O:3])[O:4][c:5]1[cH:6][cH:7][cH:8][cH:9][c:10]1[C:11](=[O:12])[OH:13]>>[CH3:1][C:2]([OH:3])=